# アメフト画像解析 バックエンド（FastAPI + YOLOv8）

このノートブックは、アメフト体験会用のAIバックエンドサーバーです。
Google Colab上でFastAPIを立ち上げ、`ngrok` または `localtunnel` を使って外部（CodeSandboxのフロントエンド）からアクセスできるようにします。

**【使い方】**
1. 上から順番にセルを実行（再生ボタン▶を押す）してください。
2. 最後のセルを実行すると、`https://xxxx.ngrok-free.app` のようなURLが表示されます。
3. そのURLをコピーして、CodeSandboxの `App.jsx` の `BACKEND_URL` に貼り付けてください。

## 1. 必要なライブラリのインストール

In [ ]:
!pip install fastapi uvicorn python-multipart ultralytics pyngrok nest-asyncio

## 2. バックエンドサーバーのコード作成
FastAPIとYOLOv8を使って、画像を受け取り解析結果を返すAPIを作ります。

In [ ]:
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
import cv2
import numpy as np
from ultralytics import YOLO
import base64
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# ==========================================
# 1. Webサーバーの準備
# ==========================================
app = FastAPI()

# フロントエンド（CodeSandbox）からの通信を許可する設定
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # どこからでもアクセスOK
    allow_methods=["*"],  # どんな通信方法でもOK
)

# ==========================================
# 2. AIモデルの準備
# ==========================================
# YOLOv8の「nano（一番軽くて速い）」モデルを読み込む
model = YOLO('yolov8n.pt')

# ==========================================
# 3. APIの作成（URLにアクセスされたときの処理）
# ==========================================

@app.get("/")
def hello():
    return {"message": "AIサーバーが動いています！"}

@app.post("/analyze")
async def analyze_image(file: UploadFile = File(...)):
    # --- A. 画像を読み込む ---
    contents = await file.read()
    nparr = np.frombuffer(contents, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # --- B. AIで物体検出 ---
    # conf=0.25 は「25%以上の自信があるものだけ見つける」という設定
    # ★体験会ポイント：ここの数字を 0.80 などに変えて、結果がどう変わるか試してみよう！
    results = model(img, conf=0.25)
    
    # --- C. 結果を画像に書き込む ---
    annotated_img = results[0].plot()
    
    # --- D. 人数を数える ---
    # クラス番号「0」が「人（person）」を表す
    person_count = 0
    for box in results[0].boxes:
        if int(box.cls) == 0:
            person_count += 1

    # --- E. フロントエンドに返す準備 ---
    _, buffer = cv2.imencode('.jpg', annotated_img)
    img_base64 = base64.b64encode(buffer).decode('utf-8')
    
    return {
        "person_count": person_count,
        "image_data": f"data:image/jpeg;base64,{img_base64}"
    }

## 3. サーバーの起動とURLの発行
ngrokを使って、Colabの中で動いているサーバーをインターネットに公開します。

※ 初回のみ、[ngrokの公式サイト](https://dashboard.ngrok.com/get-started/your-authtoken) で無料アカウントを作成し、Authtokenを取得して下のセルに入力する必要があります。

In [ ]:
import getpass

# ngrokのAuthtokenを入力（画面に入力欄が出ます）
print("ngrokのAuthtokenを入力してください:")
authtoken = getpass.getpass()
ngrok.set_auth_token(authtoken)

# ngrokでポート8000を公開
ngrok_tunnel = ngrok.connect(8000)
print("\n==================================================")
print("★ 以下のURLをコピーして、CodeSandboxの BACKEND_URL に貼り付けてください ★")
print("Public URL:", ngrok_tunnel.public_url)
print("==================================================\n")

# FastAPIサーバーを起動
nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)